# NetCDF Comparison: Diffusion Interpolator vs Reference

Side-by-side comparison of two `interpolator_MEPS_2024-01_11_memb.nc` files.

- **Left column**: `_saved/outputs/` (diffusion interpolation)
- **Right column**: `metno_interpolator/data/` (reference)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from IPython.display import display
import ipywidgets as widgets

In [ ]:
# ── File paths ──────────────────────────────────────────────────────
FILE_A = "/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_saved/outputs/interpolator_MEPS_2024-11_11_memb.nc"
FILE_A = "/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_saved/outputs_answer/interpolator_MEPS_2024-11_11_memb.nc"

FILE_B = "/project/home/p200177/DE_371/metno_interpolator/data/interpolator_MEPS_2024-11_11_memb.nc"

LABEL_A = "Diffusion Interpolation"
LABEL_B = "Reference (MetNo)"

ds_a = xr.open_dataset(FILE_A)
ds_b = xr.open_dataset(FILE_B)

print("Dataset A variables:", list(ds_a.data_vars))
print("Dataset B variables:", list(ds_b.data_vars))
print(f"Time steps: {ds_a.sizes['time']}, Ensemble members: {ds_a.sizes['ensemble']}")
print(f"Spatial grid: {ds_a.sizes['y']} x {ds_a.sizes['x']}")

## Configuration

Edit the cell below to choose which **variables**, **time indices**, and **ensemble member** to compare.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  USER CONFIGURATION — edit these to change what gets plotted
# ══════════════════════════════════════════════════════════════════════

# Variables to compare (must exist in both files)
VARIABLES = ["10u", "10v", "2t", "msl", "tp", "ws10"]

# Time indices to plot (0-based; max 240 for 241 steps)
TIME_INDICES = [3, 63, 123, 183, 240-3]

# Ensemble member index (0-based; 0–10 for 11 members)
ENSEMBLE_IDX = 1

# Use lat/lon coordinates for plotting (set False for plain y/x grid)
USE_LATLON = True

# Colormap per variable (defaults to 'RdBu_r')
CMAPS = {
    "10u": "RdBu_r",
    "10v": "RdBu_r",
    "2t": "RdYlBu_r",
    "msl": "viridis",
    "tp": "YlGnBu",
    "ws10": "magma",
}

# Pretty labels
VAR_LABELS = {
    "10u": "10m U-wind [m/s]",
    "10v": "10m V-wind [m/s]",
    "2t": "2m Temperature [K]",
    "msl": "Mean Sea-Level Pressure [Pa]",
    "tp": "Total Precipitation [m]",
    "ws10": "10m Wind Speed [m/s]",
}

## Side-by-Side Spatial Comparison

For each selected variable and time step: left = Dataset A, right = Dataset B. A shared colorbar ensures the same color scale.

In [ ]:
def get_field(ds, var, time_idx, ens_idx):
    """Extract a 2-D field from a dataset."""
    return ds[var].isel(time=time_idx, ensemble=ens_idx).values


def plot_comparison(var, time_idx, ens_idx=ENSEMBLE_IDX):
    """Plot one variable at one time step: A | B | (A − B)."""
    field_a = get_field(ds_a, var, time_idx, ens_idx)
    field_b = get_field(ds_b, var, time_idx, ens_idx)
    diff = field_a - field_b

    # Shared range for A and B
    vmin = np.nanmin([field_a, field_b])
    vmax = np.nanmax([field_a, field_b])

    # Symmetric range for difference
    dmax = np.nanmax(np.abs(diff))

    cmap = CMAPS.get(var, "RdBu_r")
    label = VAR_LABELS.get(var, var)
    time_val = ds_a["time"].values[time_idx]

    if USE_LATLON and "lat" in ds_a and "lon" in ds_a:
        lat = ds_a["lat"].values
        lon = ds_a["lon"].values
    else:
        lat = lon = None

    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

    for ax, field, title in zip(
        axes[:2],
        [field_a, field_b],
        [LABEL_A, LABEL_B],
    ):
        if lat is not None:
            im = ax.pcolormesh(lon, lat, field, vmin=vmin, vmax=vmax,
                               cmap=cmap, shading="auto")
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
        else:
            im = ax.imshow(field, origin="lower", vmin=vmin, vmax=vmax,
                           cmap=cmap, aspect="equal")
            ax.set_xlabel("x")
            ax.set_ylabel("y")
        ax.set_title(title)

    fig.colorbar(im, ax=axes[:2].tolist(), label=label, shrink=0.85)

    # Difference panel
    ax_d = axes[2]
    if lat is not None:
        im_d = ax_d.pcolormesh(lon, lat, diff, vmin=-dmax, vmax=dmax,
                               cmap="RdBu_r", shading="auto")
        ax_d.set_xlabel("Longitude")
        ax_d.set_ylabel("Latitude")
    else:
        im_d = ax_d.imshow(diff, origin="lower", vmin=-dmax, vmax=dmax,
                           cmap="RdBu_r", aspect="equal")
        ax_d.set_xlabel("x")
        ax_d.set_ylabel("y")
    ax_d.set_title(f"Difference (A − B)")
    fig.colorbar(im_d, ax=ax_d, label=f"Δ {label}", shrink=0.85)

    fig.suptitle(f"{label}  ·  time index {time_idx} ({time_val})  ·  ensemble {ens_idx}",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Generate all comparison plots ──────────────────────────────────
for var in VARIABLES:
    for tidx in TIME_INDICES:
        plot_comparison(var, tidx)

## Summary Statistics

In [ ]:
import pandas as pd

rows = []
for var in VARIABLES:
    a = ds_a[var].isel(ensemble=ENSEMBLE_IDX).values
    b = ds_b[var].isel(ensemble=ENSEMBLE_IDX).values
    diff = a - b
    rows.append({
        "Variable": VAR_LABELS.get(var, var),
        "Mean A": f"{np.nanmean(a):.4f}",
        "Mean B": f"{np.nanmean(b):.4f}",
        "Mean Diff": f"{np.nanmean(diff):.4e}",
        "RMSE": f"{np.sqrt(np.nanmean(diff**2)):.4e}",
        "Max |Diff|": f"{np.nanmax(np.abs(diff)):.4e}",
    })

stats_df = pd.DataFrame(rows)
stats_df

## Time-Series Comparison (Domain Mean)

Spatially averaged time series for each variable, overlaid for both datasets.

In [ ]:
fig, axes = plt.subplots(len(VARIABLES), 1, figsize=(14, 3.5 * len(VARIABLES)),
                         sharex=True)
if len(VARIABLES) == 1:
    axes = [axes]

for ax, var in zip(axes, VARIABLES):
    mean_a = ds_a[var].isel(ensemble=ENSEMBLE_IDX).mean(dim=["y", "x"]).values
    mean_b = ds_b[var].isel(ensemble=ENSEMBLE_IDX).mean(dim=["y", "x"]).values
    time_vals = np.arange(len(mean_a))

    ax.plot(time_vals, mean_a, label=LABEL_A, linewidth=1.5)
    ax.plot(time_vals, mean_b, label=LABEL_B, linewidth=1.5, linestyle="--")
    ax.set_ylabel(VAR_LABELS.get(var, var), fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Time index")
fig.suptitle(f"Domain-Mean Time Series  ·  Ensemble member {ENSEMBLE_IDX}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Interactive Explorer (optional)

Use the widget below to interactively browse variable / time / ensemble combinations.

In [ ]:
try:
    var_dropdown = widgets.Dropdown(options=VARIABLES, value=VARIABLES[0],
                                    description="Variable:")
    time_slider = widgets.IntSlider(value=0, min=0, max=ds_a.sizes["time"] - 1,
                                    step=1, description="Time idx:")
    ens_slider = widgets.IntSlider(value=0, min=0, max=ds_a.sizes["ensemble"] - 1,
                                   step=1, description="Ensemble:")

    ui = widgets.VBox([var_dropdown, time_slider, ens_slider])
    out = widgets.interactive_output(
        plot_comparison,
        {"var": var_dropdown, "time_idx": time_slider, "ens_idx": ens_slider},
    )
    display(ui, out)
except Exception as e:
    print(f"Interactive widgets not available ({e}). Use the static plots above.")